# DE-07 — Observability, Security & Governance

**Dataset:** `data/loan_data_07.csv`

This notebook demonstrates logs, metrics, trace context, lineage, freshness/duration/throughput/volume/quality signals, data classification, masking, tokenization, retention/deletion, least privilege, secrets, auditability, segregation of duties, monitoring queries, and an incident evidence pack.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

DATA_FILE = ROOT / "data" / "loan_data_07.csv"
assert DATA_FILE.exists(), f"Dataset not found: {DATA_FILE}"

raw = pd.read_csv(DATA_FILE)
print(f"Dataset: {DATA_FILE.name}")
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
print(raw.head(3).to_string(index=False))

## Learning Content

### Observability

- **Logs** explain discrete events with consistent context.
- **Metrics** quantify health over time.
- **Traces** connect work across pipeline stages.
- **Lineage** records where data came from and how it changed.
- Core signals include freshness, duration, throughput, volume, and quality.

### Security and governance

- Classification determines required handling.
- Encryption protects data at rest and in transit.
- Masking and tokenization reduce exposure.
- Retention and deletion enforce lifecycle obligations.
- Least privilege limits capabilities.
- Secrets belong in managed secret storage.
- Auditability records who did what and when.
- Segregation of duties prevents one identity from controlling every critical action.

In [ ]:
import hashlib
import json
import logging
import uuid
from datetime import datetime, timezone
from time import perf_counter

run_id = str(uuid.uuid4())
trace_id = uuid.uuid4().hex
started_at = datetime.now(timezone.utc)
timer = perf_counter()

logger = logging.getLogger("de07")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
    logger.addHandler(handler)

logger.info(json.dumps({
    "event": "pipeline_started",
    "run_id": run_id,
    "trace_id": trace_id,
    "source": DATA_FILE.name,
}))

In [ ]:
# Process and collect operational signals.
processed = raw.copy()
processed["TotalIncome"] = processed["ApplicantIncome"] + processed["CoapplicantIncome"]
processed["observed_at"] = started_at

duration_seconds = perf_counter() - timer
metrics = {
    "freshness_seconds": 0.0,  # delivery timestamp is simulated as now for this CSV demo
    "duration_seconds": duration_seconds,
    "throughput_rows_per_second": len(processed) / max(duration_seconds, 1e-9),
    "volume_rows": len(processed),
    "quality_valid_status_ratio": float(processed["Loan_Status"].isin(["Y", "N"]).mean()),
}

lineage = pd.DataFrame([
    [DATA_FILE.name, "bronze.loan_applications", "read + provenance"],
    ["bronze.loan_applications", "silver.loan_applications", "type + standardize + derive TotalIncome"],
    ["silver.loan_applications", "gold.loan_monitoring_summary", "aggregate"],
], columns=["upstream", "downstream", "transformation"])

logger.info(json.dumps({
    "event": "pipeline_completed",
    "run_id": run_id,
    "trace_id": trace_id,
    "rows": len(processed),
    "duration_seconds": duration_seconds,
}))

print("Metrics:", metrics)
print("\nLineage:\n", lineage.to_string(index=False))

## Hands-on / Demonstration

### Classification, masking, tokenization, retention, and deletion

The dataset does not contain names or email addresses. `Loan_ID` is still treated as a confidential identifier.

Hashing below is **one-way tokenization, not encryption**. Production encryption should use an approved cryptographic library and keys managed by KMS/HSM. PostgreSQL connections should require TLS, and storage encryption should be managed by the platform.

In [ ]:
classification = pd.DataFrame([
    ["Loan_ID", "Confidential identifier", "Tokenize outside trusted zone"],
    ["ApplicantIncome", "Confidential financial", "Aggregate or mask"],
    ["CoapplicantIncome", "Confidential financial", "Aggregate or mask"],
    ["Credit_History", "Restricted decision attribute", "Role-based access"],
    ["Loan_Status", "Internal", "Approved analytics use"],
], columns=["column", "classification", "handling"])

def tokenise(value: str) -> str:
    # Training-only deterministic token. Production should use a keyed HMAC
    # or approved tokenization service to resist dictionary attacks.
    return hashlib.sha256(value.encode("utf-8")).hexdigest()[:16]

secured = processed.copy()
secured["Loan_ID_Token"] = secured["Loan_ID"].map(tokenise)
secured["ApplicantIncome_Band"] = pd.cut(
    secured["ApplicantIncome"],
    bins=[-np.inf, 3000, 6000, np.inf],
    labels=["Low", "Medium", "High"],
)
consumer_view = secured[[
    "Loan_ID_Token", "ApplicantIncome_Band",
    "Property_Area", "Loan_Status",
]]

# Simulated retention: records older than the cutoff are deletion candidates.
secured["retention_date"] = started_at + pd.Timedelta(days=90)
retention_cutoff = started_at + pd.Timedelta(days=91)
deletion_candidates = secured[secured["retention_date"] < retention_cutoff]
retained = secured[secured["retention_date"] >= retention_cutoff]

print(classification.to_string(index=False))
print("\nMasked consumer view:\n", consumer_view.head(3).to_string(index=False))
print("\nDeletion candidates:", len(deletion_candidates), "| Retained:", len(retained))

### Least privilege, secrets, auditability, and segregation of duties

In [ ]:
access_matrix = pd.DataFrame([
    ["pipeline_runtime", "Bronze/Silver/Gold", "read source; write pipeline tables", "Service owner"],
    ["data_analyst", "Approved Gold views", "read only", "Analytics manager"],
    ["security_admin", "Roles and audit policy", "administer access; no business approval", "Security"],
    ["data_owner", "Quality and release approval", "approve; no runtime credential", "Loan Operations"],
], columns=["role", "scope", "permission", "approval_owner"])

# This confirms only whether a secret is configured; it never prints the value.
import os
secret_is_configured = bool(os.getenv("AUDIT_ENCRYPTION_KEY"))
audit_log = pd.DataFrame([{
    "run_id": run_id,
    "trace_id": trace_id,
    "actor": "pipeline_runtime",
    "action": "publish_monitoring_summary",
    "recorded_at": datetime.now(timezone.utc),
    "row_count": len(processed),
    "status": "SUCCESS",
}])

assert not access_matrix["permission"].str.contains("all permissions", case=False).any()
print(access_matrix.to_string(index=False))
print("\nSecret configured:", secret_is_configured)
print("\nAudit event:\n", audit_log.to_string(index=False))

### Build monitoring queries

These DataFrame queries represent the same questions that SQL monitoring views should answer.

In [ ]:
monitoring_runs = pd.DataFrame([
    {
        "run_id": run_id,
        "status": "SUCCESS",
        "started_at": started_at,
        "duration_seconds": metrics["duration_seconds"],
        "row_count": metrics["volume_rows"],
        "quality_ratio": metrics["quality_valid_status_ratio"],
        "owner": "Data Engineering",
    },
    {
        "run_id": "previous-training-run",
        "status": "FAILED",
        "started_at": started_at - pd.Timedelta(days=1),
        "duration_seconds": 75.0,
        "row_count": 0,
        "quality_ratio": 0.0,
        "owner": "Data Engineering",
    },
])

failed_runs = monitoring_runs[monitoring_runs["status"] == "FAILED"]
sla_breaches = monitoring_runs[monitoring_runs["duration_seconds"] > 60]
volume_anomalies = monitoring_runs[monitoring_runs["row_count"] < 1]

print("Failed runs:\n", failed_runs.to_string(index=False))
print("\nSLA breaches:\n", sla_breaches.to_string(index=False))
print("\nVolume anomalies:\n", volume_anomalies.to_string(index=False))

### Create an incident evidence pack

An evidence pack should be safe to share with responders: include identifiers, timestamps, impact, owner, and recovery guidance—but exclude credentials and sensitive row values.

In [ ]:
incident_evidence_pack = {
    "incident_id": "INC-TRAINING-001",
    "what_failed": "Previous training run published zero rows",
    "where": "Gold monitoring summary",
    "since_when": str(failed_runs.iloc[0]["started_at"]),
    "impact": "Loan dashboard may be stale",
    "owner": failed_runs.iloc[0]["owner"],
    "trace_id": "trace-not-available-for-simulated-run",
    "evidence": {
        "status": failed_runs.iloc[0]["status"],
        "row_count": int(failed_runs.iloc[0]["row_count"]),
        "duration_seconds": float(failed_runs.iloc[0]["duration_seconds"]),
    },
    "safe_next_action": "Validate source delivery and replay the bounded idempotent run",
}

print(json.dumps(incident_evidence_pack, indent=2))

## Enterprise Control

Monitoring should answer:

1. **What** failed?
2. **Where** did it fail?
3. **Since when** has the condition existed?
4. What is the **impact**?
5. Who is the **owner**?
6. What is the **safe next action**?

Security evidence should also show classification, access approval, secret handling, audit events, retention execution, and separation of runtime versus approval duties.

In [ ]:
required_incident_fields = {
    "what_failed", "where", "since_when",
    "impact", "owner", "safe_next_action",
}
assert required_incident_fields.issubset(incident_evidence_pack)
assert metrics["volume_rows"] == len(raw)
assert metrics["quality_valid_status_ratio"] == 1.0
assert consumer_view["Loan_ID_Token"].is_unique
assert "Loan_ID" not in consumer_view.columns

print("DE-07 controls passed.")